In [0]:
storage_account = "dmpro2"
access_key = "blkSylna0g9RsyAQUhzDDyJZ4fS4oWfrveP8HO+FYhp0Y1Cv6hcIGjo2UmS/MWA/TWcTWiAHRTy5+AStQPiZYA=="   # hard-coded

spark.conf.set(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", access_key)

In [0]:
from pyspark.sql.functions import *


catalog_name = "dm_pro2"
base_path = "abfss://rawdata@dmpro2.dfs.core.windows.net/data"


# Read Bronze as a stream
orders_bronze_stream = (spark.readStream
    .format("delta")
    .table(f"{catalog_name}.bronze.orders"))


# Transform
orders_silver_stream = (orders_bronze_stream
    .withColumn("OrderDate", to_date(col("OrderDate"), "yyyy-MM-dd"))
    .withColumn("Quantity", col("Quantity").cast("int"))
    .withColumn("TotalAmount", col("TotalAmount").cast("double"))
    .withColumn("Status", initcap(trim(col("Status"))))
    .dropna(subset=["OrderID", "CustomerID", "ProductID"])
)


# Write to Silver incrementally
(orders_silver_stream.writeStream
    .format("delta")
    .option("checkpointLocation", f"{base_path}/checkpoints/orders_silver")
    .outputMode("append")
    .trigger(availableNow=True)
    .table(f"{catalog_name}.silver.orders"))

returns_bronze_stream = (spark.readStream
    .format("delta")
    .table(f"{catalog_name}.bronze.returns"))


returns_silver_stream = (returns_bronze_stream
    .withColumn("ReturnDate", to_date(col("ReturnDate"), "yyyy-MM-dd"))
    .withColumn("RefundAmount", col("RefundAmount").cast("double"))
    .dropna(subset=["ReturnID", "OrderID"])
)


(returns_silver_stream.writeStream
    .format("delta")
    .option("checkpointLocation", f"{base_path}/checkpoints/returns_silver")
    .outputMode("append")
    .trigger(availableNow=True)
    .table(f"{catalog_name}.silver.returns"))


In [0]:
%sql
select * from dm_pro2.silver.orders;

OrderID,CustomerID,ProductID,OrderDate,Quantity,TotalAmount,Status,_source_file,ingested_at
O001,C001,P001,2025-11-01,1,699.0,Completed,abfss://rawdata@dmpro2.dfs.core.windows.net/data/orders/orders_2025_11_01.csv,2025-12-01T04:54:54.97Z
O021,C999,P001,2025-11-01,1,699.0,Completed,abfss://rawdata@dmpro2.dfs.core.windows.net/data/orders/orders_2025_11_01.csv,2025-12-01T04:54:54.97Z
O026,C002,P004,2025-11-01,2,178.0,Shipped,abfss://rawdata@dmpro2.dfs.core.windows.net/data/orders/orders_2025_11_01.csv,2025-12-01T04:54:54.97Z
O027,C003,P002,2025-11-01,1,1299.0,Completed,abfss://rawdata@dmpro2.dfs.core.windows.net/data/orders/orders_2025_11_01.csv,2025-12-01T04:54:54.97Z
O028,C004,P005,2025-11-01,1,199.0,Completed,abfss://rawdata@dmpro2.dfs.core.windows.net/data/orders/orders_2025_11_01.csv,2025-12-01T04:54:54.97Z
O029,C005,P008,2025-11-01,1,249.0,Pending,abfss://rawdata@dmpro2.dfs.core.windows.net/data/orders/orders_2025_11_01.csv,2025-12-01T04:54:54.97Z
O030,C006,P013,2025-11-01,3,75.0,Completed,abfss://rawdata@dmpro2.dfs.core.windows.net/data/orders/orders_2025_11_01.csv,2025-12-01T04:54:54.97Z
O031,C007,P009,2025-11-01,2,258.0,Shipped,abfss://rawdata@dmpro2.dfs.core.windows.net/data/orders/orders_2025_11_01.csv,2025-12-01T04:54:54.97Z
O032,C008,P016,2025-11-01,1,199.0,Completed,abfss://rawdata@dmpro2.dfs.core.windows.net/data/orders/orders_2025_11_01.csv,2025-12-01T04:54:54.97Z
O033,C009,P019,2025-11-01,1,999.0,Completed,abfss://rawdata@dmpro2.dfs.core.windows.net/data/orders/orders_2025_11_01.csv,2025-12-01T04:54:54.97Z
